In [1]:
!pip install Cohere faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.0 MB/s eta 0:00:00


In [4]:
import cohere
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

#The Sentence Transformers library (by UKP Lab, built on top of Hugging
#Face & PyTorch) is designed to make it easy to create sentence,
#paragraph, and document embeddings for semantic tasks.

In [5]:
docs = ["Steps to configure email signature in Outlook.",

"How to reset your company email password securely.",

"Creating folders and filters in your inbox.",

"Understanding company IT policy for account security.",

"How to request a new laptop from IT department.",

"How to download an app from Google App store",

"Summary of emails received",

"What is global warming",

"How to have a balanced diet",

"How to login in gmail account",

"All about corporate emails",

"Understanding you email account",

"Help with company email",

"All about managing company email accounts",

"Help with resetting password of email account"
]

In [ ]:
co = cohere.Client("API_KEY")

embedder = SentenceTransformer("all-miniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
doc_embeddings = embedder.encode(sentences=docs, convert_to_numpy=True)

dimension=doc_embeddings.shape[1]
faiss_index=faiss.IndexFlatL2(dimension)

# Adding embeddings to the vector database FAISS
faiss_index.add(doc_embeddings)

In [15]:
# User's query
query = ["How do I reset my work email password?"]

# Embedding the user's query
query_embedding = embedder.encode(sentences=query, convert_to_numpy=True)
distance, indices = faiss_index.search(query_embedding, k=5)

retrieved_docs = [docs[i] for i in indices[0]]

In [17]:
# Available models rerank-v3.5, rerank-english-v3.0, rerank-multilingual-v3.0
rerank_response = co.rerank(query=query[0], documents=retrieved_docs, model="rerank-v3.5")

for i, result in enumerate(rerank_response.results, 1):
  print(f"{retrieved_docs[result.index]} \nImportance Score: {result.relevance_score}\n")

[RerankResponseResultsItem(document=None, index=1, relevance_score=0.5441965), RerankResponseResultsItem(document=None, index=0, relevance_score=0.5068117), RerankResponseResultsItem(document=None, index=3, relevance_score=0.20154628), RerankResponseResultsItem(document=None, index=2, relevance_score=0.13828741), RerankResponseResultsItem(document=None, index=4, relevance_score=0.13286321)]
How to reset your company email password securely. 
Importance Score: 0.5441965

Help with resetting password of email account 
Importance Score: 0.5068117

Help with company email 
Importance Score: 0.20154628

How to login in gmail account 
Importance Score: 0.13828741

All about managing company email accounts 
Importance Score: 0.13286321

